In [2]:
import rasterio
import numpy as np
import os
from rasterio.enums import Resampling

# --- 📌 Localisation des fichiers Landsat ---
base_path = r"C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT"
prefix = "HLS.L30.T40QFJ.2025043T062900.v2.0"

# --- 📌 Définition des combinaisons ---
band_combinations = {
    "NaturalColor_432": (4, 3, 2),
    "FalseColor_543": (5, 4, 3),
    "Geological_753": (7, 5, 3),
    "Geological_bis_347": (3, 4, 7),
}

# --- 📌 Chargement des bandes Landsat ---
bands = {}
nodata_value = -9999.0  # Valeur nodata à corriger

for i in range(1, 8):  # Bandes 1 à 7
    file_path = os.path.join(base_path, f"{prefix}.B0{i}.TIF")
    if os.path.exists(file_path):
        with rasterio.open(file_path) as src:
            band_data = src.read(1, resampling=Resampling.nearest).astype(np.float32)
            
            # Remplacement des valeurs nodata
            band_data[band_data == nodata_value] = 0  # Ou np.nan si tu veux ignorer ces valeurs
            
            bands[i] = band_data
            profile = src.profile  # Métadonnées pour l'export
    else:
        print(f"⚠️ Fichier manquant : {file_path}")

# --- 📌 Génération des GeoTIFF ---
for name, (r, g, b) in band_combinations.items():
    if r in bands and g in bands and b in bands:
        rgb_image = np.stack([bands[r], bands[g], bands[b]], axis=0)

        # Normalisation des valeurs entre 0 et 65535 (uint16)
        min_val, max_val = np.nanmin(rgb_image), np.nanmax(rgb_image)
        rgb_image = ((rgb_image - min_val) / (max_val - min_val) * 65535).astype(np.uint16)

        # Mise à jour du profil pour 3 bandes
        profile.update(count=3, dtype=rasterio.uint16, nodata=0)

        output_file = os.path.join(base_path, f"{prefix}_{name}.tif")
        with rasterio.open(output_file, "w", **profile) as dst:
            dst.write(rgb_image)

        print(f"✅ Image {output_file} générée ({r}-{g}-{b})")
    else:
        print(f"❌ Impossible de générer {name}, une ou plusieurs bandes sont manquantes.")

print("🎉 Traitement terminé !")


⚠️ Fichier manquant : C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0.B06.TIF
✅ Image C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0_NaturalColor_432.tif générée (4-3-2)
✅ Image C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0_FalseColor_543.tif générée (5-4-3)
✅ Image C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0_Geological_753.tif générée (7-5-3)
✅ Image C:\Users\scolzy\Stage_SC_26\OMAN_Rasters_test\05_TELEDETECTION\01_LANDSAT\HLS.L30.T40QFJ.2025043T062900.v2.0_Geological_bis_347.tif générée (3-4-7)
🎉 Traitement terminé !
